<a href="https://colab.research.google.com/github/andyLiSyr/Stout-Take-Home-Case-Study/blob/main/AndyLi_Stout_CaseStudy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load CSV into a Pandas Dataframe

In [1]:
import pandas as pd
import sqlite3
from google.colab import files

# Please run in Google Colab Notebook to reproduce output

# Upload CSV file from local device from the Choose Files Widget
uploaded = files.upload()

# Get uploaded filename
csv_file = next(iter(uploaded))

# Load CSV into pandas
df = pd.read_csv(csv_file)



Saving casestudy.csv to casestudy.csv


Quick Data Cleaning on Customer Email field and Data Quality Checks

In [2]:
# Noticed leading whitespace in customer emails when previewing the CSV.
# Trim whitespace to ensure identical emails are treated as the same customer
df['customer_email'] = df['customer_email'].str.strip()

# check for nulls
total_nulls=(df.isnull().sum())
print(f"Total null values: {total_nulls}")
print("")


Total null values: Unnamed: 0        0
customer_email    0
net_revenue       0
year              0
dtype: int64



In [3]:
#check year values
display(sorted(df['year'].unique()))

#check revenue values
display(df['net_revenue'].describe())

[np.int64(2015), np.int64(2016), np.int64(2017)]

,net_revenue
count,685927.000000
mean,125.647755
std,71.854528
min,1.000000
25%,63.510000
50%,125.730000
75%,187.820000
max,250.000000


Load Pandas Dataframe into SQLite DB to write SQL queries

In [4]:
# Create SQLite database
conn = sqlite3.connect("customers.db")

# Load DataFrame into SQLite table
table_name = "customer_orders"
df.to_sql(table_name, conn, if_exists="replace", index=False)

# Test SQL query
query = """
SELECT *
FROM customer_orders
LIMIT 10;
"""

results = pd.read_sql_query(query, conn)

# Display results
display(results)

# Print row count
print(f"Successfully loaded {len(df)} rows into SQLite.")

,Unnamed: 0,customer_email,net_revenue,year
0,0,nhknapwsbx@gmail.com,249.92,2015
1,1,joiuzbvcpn@gmail.com,87.61,2015
2,2,ukkjctepxt@gmail.com,168.38,2015
3,3,gykatilzrt@gmail.com,62.40,2015
4,4,mmsgsrtxah@gmail.com,43.08,2015
5,5,mobvusnzfr@gmail.com,39.44,2015
6,6,kxwkqvacsv@gmail.com,171.46,2015
7,7,iqmnugdcjs@gmail.com,111.99,2015
8,8,baiikostmd@gmail.com,170.70,2015
9,9,lfeafnigbu@gmail.com,54.99,2015


Successfully loaded 685927 rows into SQLite.


For these SQL Queries for the 10 questions for this assignment, I will look at the data with the current year being 2016. I have created parameter in the next cell, so if a user wanted to look at a different year they can just change the value of the parameter variable

In [5]:
#Create current year variable that reference the current year to query
current_year = 2016

Question 1
Total Revenue for the current year

In [6]:
#Question 1
#Total Revenue
query = """
SELECT SUM(net_revenue) AS Total_Revenue_Current_Year
FROM customer_orders
WHERE year=:current_year;
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year

    }
)

# Display results
display(results)

,Total_Revenue_Current_Year
0,25730943.59


Question 2
New Customer Revenue

In [7]:
#Question 2
#New Customer Revenue
query = """
SELECT SUM(net_revenue) AS New_Customer_Revenue
FROM customer_orders
WHERE year = :current_year
  AND customer_email NOT IN (
      SELECT customer_email
      FROM customer_orders
      WHERE year = :current_year-1
  );
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year
    }
)

# Display results
display(results)

,New_Customer_Revenue
0,17206366.9


Question 3
Existing Customer Growth

In [8]:
#Question 3
#Existing Customer Growth
query = """
SELECT SUM(curr_year.net_revenue-prev_year.net_revenue) AS Existing_Customer_Growth
FROM customer_orders AS curr_year
JOIN customer_orders AS prev_year
ON curr_year.customer_email=prev_year.customer_email AND curr_year.year= :current_year AND prev_year.year= :current_year-1
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year

    }
)

# Display results
display(results)

,Existing_Customer_Growth
0,39043.65


Question 4
Revenue Lost From Attrition

In [9]:
#Question 4
#Revenue Lost From Attrition
query = """
SELECT SUM(net_revenue) AS Revenue_Lost_From_Attrition
FROM customer_orders
WHERE year = :current_year-1
  AND customer_email NOT IN (
      SELECT customer_email
      FROM customer_orders
      WHERE year = :current_year
  );
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year
    }
)

# Display results
display(results)

,Revenue_Lost_From_Attrition
0,20551216.15


Question 5
Existing Customer Revenue Current Year

In [10]:
#Question 5
#Existing Customer Revenue Current Year
query = """
SELECT SUM(curr_year.net_revenue) AS Existing_Customer_Revenue_Current_Year
FROM customer_orders AS curr_year
JOIN customer_orders AS prev_year
ON curr_year.customer_email=prev_year.customer_email AND curr_year.year= :current_year AND prev_year.year= :current_year-1
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year

    }
)

# Display results
display(results)

,Existing_Customer_Revenue_Current_Year
0,8524576.69


Question 6
Existing Customer Revenue Prior Year

In [11]:
#Question 6
#Existing Customer Revenue Prior Year
query = """
SELECT SUM(prev_year.net_revenue) AS Existing_Customer_Revenue_Prior_Year
FROM customer_orders AS curr_year
JOIN customer_orders AS prev_year
ON curr_year.customer_email=prev_year.customer_email AND curr_year.year= :current_year AND prev_year.year= :current_year-1
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year

    }
)

# Display results
display(results)

,Existing_Customer_Revenue_Prior_Year
0,8485533.04


Question 7
Total Customers Current Year

In [12]:
#Question 7
#Total Customers Current Year
query = """
SELECT COUNT (DISTINCT customer_email) AS Total_Customers_Current_Year
FROM customer_orders
WHERE year=:current_year;
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year

    }
)

# Display results
display(results)

,Total_Customers_Current_Year
0,204646


Question 8
Total Customers Previous Year

In [13]:
#Question 8
#Total Customers Previous Year
query = """
SELECT COUNT(DISTINCT customer_email) AS Total_Customers_Previous_Year
FROM customer_orders
WHERE year=:current_year-1
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year
    }
)

# Display results
display(results)

,Total_Customers_Previous_Year
0,231294


Question 9
New Customers

In [14]:
#Question 9
#New Customers
query = """
SELECT COUNT(DISTINCT(customer_email)) AS New_Customers
FROM customer_orders
WHERE year = :current_year
  AND customer_email NOT IN (
      SELECT customer_email
      FROM customer_orders
      WHERE year = :current_year-1
  );
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year
    }
)

# Display results
display(results)

,New_Customers
0,136891


Question 10
Lost Customers

In [15]:
#Question 10
#Lost Customers
query = """
SELECT COUNT(DISTINCT(customer_email)) AS Lost_Customers
FROM customer_orders
WHERE year = :current_year-1
  AND customer_email NOT IN (
      SELECT customer_email
      FROM customer_orders
      WHERE year = :current_year
  );
"""

results = pd.read_sql_query(
    query,
    conn,
    params={
        "current_year": current_year
    }
)

# Display results
display(results)

,Lost_Customers
0,163539


In [16]:
conn.close()